# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and initialize Dataset object
dataset = mlc.Dataset(croissant_url)

# Print out some basic dataset metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Data Collection: {getattr(meta, 'dataCollection', None)}")

## 2. Data Overview
Review available record sets, their fields, and column details. All entities are referenced by their `@id`.

In [ ]:
# List available record sets and their details
record_sets = dataset.metadata.recordSets
print("Available Record Sets (with @id, name):")
for record_set in record_sets:
    rs_id = record_set['@id'] if isinstance(record_set, dict) else getattr(record_set, '@id', None)
    rs_name = record_set['name'] if isinstance(record_set, dict) and 'name' in record_set else getattr(record_set, 'name', None)
    print(f"  - @id: {rs_id}")
    if rs_name:
        print(f"    Name: {rs_name}")
    print("    Fields:")
    for field in getattr(record_set, 'fields', []):
        field_id = field['@id'] if isinstance(field, dict) else getattr(field, '@id', None)
        field_name = field['name'] if isinstance(field, dict) and 'name' in field else getattr(field, 'name', None)
        print(f"      - @id: {field_id}", end='')
        if field_name:
            print(f" (name: {field_name})")
        else:
            print()
    if hasattr(record_set, 'columns') or (isinstance(record_set, dict) and 'columns' in record_set):
        columns = record_set.columns if hasattr(record_set, 'columns') else record_set['columns']
        print("    Columns:")
        for col in columns:
            cid = col['@id'] if isinstance(col, dict) else getattr(col, '@id', None)
            cname = col['name'] if isinstance(col, dict) and 'name' in col else getattr(col, 'name', None)
            print(f"      - @id: {cid}", end='')
            if cname:
                print(f" (name: {cname})")
            else:
                print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all available record set IDs
record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None) for rs in dataset.metadata.recordSets]
print(f"Found record sets: {record_set_ids}")

# Load records for each record set and store as DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set @id={record_set_id} [shape={df.shape}]")
        else:
            print(f"No records found for @id={record_set_id}")
    except Exception as e:
        print(f"Could not load records for @id={record_set_id}: {e}")

# Show the first DataFrame's columns and head
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"\nFirst record set with data: {first_record_set_id}")
    print("Columns:", dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head(5))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Use the first available DataFrame and try to determine a numeric field
chosen_record_set_id = None
for rs_id, df in dataframes.items():
    # Try to select a numeric column
    numeric_candidate = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidate = col
            break
    if numeric_candidate:
        chosen_record_set_id = rs_id
        break

if chosen_record_set_id is None:
    raise ValueError("No numeric columns found in loaded DataFrames.")

numeric_field_id = numeric_candidate

print(f"Using record set @id={chosen_record_set_id} and numeric field: {numeric_field_id}\n")

# Filtering
threshold = dataframes[chosen_record_set_id][numeric_field_id].mean() if not np.isnan(dataframes[chosen_record_set_id][numeric_field_id].mean()) else 0
filtered_df = dataframes[chosen_record_set_id][dataframes[chosen_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows")
display(filtered_df.head())

# Normalization
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Try grouping by a categorical field if one is present
group_field = None
for col in filtered_df.columns:
    if col != numeric_field_id and not np.issubdtype(filtered_df[col].dtype, np.number):
        group_field = col
        break

if group_field:
    print(f"\nGrouping by '{group_field}' and showing means of numeric fields:")
    grouped = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped.head())
else:
    print("No categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in filtered_df:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a group field is found, do a boxplot
    if group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key Findings:**
- The dataset describes protein abundances and characteristics measured via mass spectrometry from human mast cell extracellular vesicle samples.
- Loaded data from record sets shows rich numeric and categorical fields, such as protein intensities and names.
- Initial EDA and visualizations demonstrate how to filter, normalize, and group protein-level data by chosen attributes.

Continue with advanced analytics or modeling as needed for your use case.